# N102 · 完全、多重与分组背包

**目标：** 根据可重复次数和对象顺序选择转移与循环顺序。

**先修：** N101, N098。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 正序容量；有限次数；二进制分组；分组选择；组合/排列计数。

## 从问题到算法

完全背包允许重复选一件，所以同一层容量要正序。计数时循环顺序决定语义：先枚举物品再金额得到无序组合；先金额再末项得到有序序列。有限次数背包可以把数量拆为1、2、4…和余数的小包，再当0/1物品处理。分组背包必须读取上一组，不能在本组内部重复选。

## 最小实现

**本章接口：** `coin_change(coins, amount)`、`coin_change_combinations(coins, amount)`、`ordered_sum_count(nums, target)`、`bounded_knapsack(items, capacity)`

In [1]:
def _positive_unique(values):
    if any(x<=0 for x in values): raise ValueError('strictly positive values required')
    return sorted(set(values))

def coin_change(coins,amount):
    coins=_positive_unique(coins); dp=[0]+[float('inf')]*amount
    for a in range(1,amount+1):
        for c in coins:
            if c<=a: dp[a]=min(dp[a],dp[a-c]+1)
    return -1 if dp[amount]==float('inf') else dp[amount]

def coin_change_combinations(coins,amount):
    dp=[1]+[0]*amount
    for c in _positive_unique(coins):
        for a in range(c,amount+1): dp[a]+=dp[a-c]
    return dp[amount]

def ordered_sum_count(nums,target):
    nums=_positive_unique(nums); dp=[1]+[0]*target
    for a in range(1,target+1):
        dp[a]=sum(dp[a-x] for x in nums if x<=a)
    return dp[target]

def bounded_knapsack(items,capacity):
    # items: (positive weight, value, nonnegative count)
    dp=[0]*(capacity+1)
    for weight,value,count in items:
        if weight<=0 or count<0: raise ValueError('invalid bounded item')
        bundle=1
        while count:
            take=min(bundle,count); w=weight*take; v=value*take
            for c in range(capacity,w-1,-1): dp[c]=max(dp[c],dp[c-w]+v)
            count-=take; bundle*=2
    return dp[-1]

def group_knapsack(groups,capacity):
    dp=[0]*(capacity+1)
    for group in groups:
        prev=dp; dp=prev[:]
        for w,v in group:
            for c in range(w,capacity+1): dp[c]=max(dp[c],prev[c-w]+v)
    return dp[-1]

## 正确性、前提与复杂度

无序组合按最后一种可使用的币值分组，不会把同一组硬币的排列重复计数；有序序列按最后一个数划分。二进制数量包的子集能表达0..原数量的每个取用量；有重复表达也不影响取最大值，但不能直接用于方案数计数。

**代价：** 完全背包O(nA)时间O(A)空间；有限次数O(C∑log(q+1))时间O(C)空间；分组O(C∑组内物品数)。amount/target/capacity非负，正值条件防止无限方案与循环依赖。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['目标','[1,2]组合数','[1,2]有序序列数'],[(a,coin_change_combinations([1,2],a),ordered_sum_count([1,2],a)) for a in range(7)])

目标,"[1,2]组合数","[1,2]有序序列数"
0,1,1
1,1,1
2,2,2
3,2,3
4,3,5
5,3,8
6,4,13


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert coin_change_combinations([1,2,5],5)==4
assert ordered_sum_count([1,2],3)==3 and coin_change_combinations([1,2],3)==2
assert coin_change([2],3)==-1 and coin_change([],0)==0
assert group_knapsack([[(1,5),(1,6)]],2)==6
from itertools import product
from random import Random
rng=Random(102)
for _ in range(150):
    items=[(rng.randrange(1,5),rng.randrange(1,8),rng.randrange(5)) for _ in range(3)]; cap=rng.randrange(12)
    feasible=[sum(t*v for t,(w,v,q) in zip(counts,items)) for counts in product(*(range(q+1) for w,v,q in items)) if sum(t*w for t,(w,v,q) in zip(counts,items))<=cap]
    assert bounded_knapsack(items,cap)==max(feasible)
for a in range(12):
    assert coin_change_combinations([1,2,5],a)==sum(x+2*y+5*z==a for x in range(a+1) for y in range(a+1) for z in range(a+1))
print('N102: 所有本章断言通过')

N102: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 手写分组背包小例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较硬币组合数与有序方案数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 322. Coin Change（canonical）
- 518. Coin Change II（canonical）
- 377. Combination Sum IV（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。